# The Idea is to get proper test dataset (pre Pca)

There is apperantely 20% overlap with the train set

In [2]:
import os
import sys

# 1. Change the working directory to the project root
%cd ..

# 2. Add the root directory to the Python path so imports work
sys.path.append(os.getcwd())

# 3. Verify we are in the right place
print(f"Current Working Directory: {os.getcwd()}")
# !ls # Optional: list files to confirm you see 'main.py' and 'pbi_utils'

/data/pavel.degterev/pbi
Current Working Directory: /data/pavel.degterev/pbi


In [3]:
import yaml
import torch
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F

from sklearn.model_selection import KFold, train_test_split
from pbi_utils.data_manager import H5pyEmbeddingsManager, PerphectDataInput, EmbeddingsManager
from main import parse_config, Stats, make_dataset, create_embeddings_bacteria, create_embeddings_phages, logger

from pbi_utils.types import CACHED_EMBEDDINGS_OPTION # bool | Literal["auto"]

In [ ]:
# just in case, this is how df is saved: (may be for custom shapley embeddings)
#
# from sklearn.model_selection import train_test_split
# train, test = train_test_split(dataset, test_size=0.2, random_state=42, shuffle=True)
# torch.save(test, config.test_path)

## initialization

In [4]:
CONFIG_PATH = "model_configs/best_model_XAI.yaml"
config = parse_config(CONFIG_PATH)
DEVICE = config.device

[DEBUG] [NT2] Using InstaDeepAI default model weights


/home/pavel.degterev/miniforge3/envs/pbi/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


[DEBUG] [NT2] Max sequence length for Nucleotide Transformer: 12276
[DEBUG] [MegaDNA] Max sequence length for megaDNA: 131071
[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 32768
[DEBUG] [NT2] Using InstaDeepAI default model weights
[DEBUG] [NT2] Max sequence length for Nucleotide Transformer: 12276
[DEBUG] [MegaDNA] Max sequence length for megaDNA: 131071
[DEBUG] [DNABERT2] Max sequence length for DNABERT2: 32768
[INFO] Configuration loaded from model_configs/best_model_XAI.yaml: Config(input_perphect=bacteria_df='data/perphect-data/all/bacteria_df.csv' phages_df='data/perphect-data/all/phages_df.csv' couples_df='data/perphect-data/all-private-oversampled/couples_df.csv', embeddings_dir=data/embeddings, num_gpu=1, gpu_id=0, training_config=TrainingConfig(do_train=True epochs=100 batch_size=256 learning_rate=0.001 weight_decay=0.0001 k_folds_cv=10 patience_early_stopping=1000 monitor_metric_early_stopping='f1' patience_reduce_lr=1000 monitor_metric_reduce_lr='f1' multiplying_fact

### Confighuration
With ability to run only those changes that actually needed

In [5]:
# I guess that here should be custom embeddings folder to not mess up OG embeddings
CUSTOM_EMBEDDINGS_FOLDER = "XAI/embeddings"
#output_manager = H5pyEmbeddingsManager(CUSTOM_EMBEDDINGS_FOLDER)
output_manager = H5pyEmbeddingsManager(config.embeddings_dir)

[INFO] Embeddings will be stored or read from data/embeddings


In [ ]:
CUSTOM_DATA_FOLDER = ''
config.input_perphect = CUSTOM_DATA_FOLDER

In [7]:
PAIR_LIST = [
    {"bacterium_id": 153, "phage_id": 2011},
    {"bacterium_id": 153, "phage_id": 2014},  # this was found in test dataset
    {"bacterium_id": 153, "phage_id": 2127},
    {"bacterium_id": 153, "phage_id": 2153},  # this was found in test dataset
    {"bacterium_id": 153, "phage_id": 2402},
    {"bacterium_id": 153, "phage_id": 2539},

    {"bacterium_id": 1869, "phage_id": 2061},
    {"bacterium_id": 1869, "phage_id": 2107},
    {"bacterium_id": 1869, "phage_id": 2332},
    {"bacterium_id": 1869, "phage_id": 5922},
    {"bacterium_id": 1869, "phage_id": 6008},
    {"bacterium_id": 1869, "phage_id": 6187},  # this was found in test dataset

    {"bacterium_id": 5787, "phage_id": 2401},
    {"bacterium_id": 5787, "phage_id": 2702},
    {"bacterium_id": 5787, "phage_id": 3048},
    {"bacterium_id": 5787, "phage_id": 3287},
    {"bacterium_id": 5787, "phage_id": 5614},
    {"bacterium_id": 5787, "phage_id": 6021},

    {"bacterium_id": 5859, "phage_id": 2051},
    {"bacterium_id": 5859, "phage_id": 2052},
    {"bacterium_id": 5859, "phage_id": 2484},
    {"bacterium_id": 5859, "phage_id": 2485},
    {"bacterium_id": 5859, "phage_id": 430},
    {"bacterium_id": 5859, "phage_id": 433},
    {"bacterium_id": 5859, "phage_id": 447},
    {"bacterium_id": 5859, "phage_id": 451},   # this was found in test dataset
    {"bacterium_id": 5859, "phage_id": 456},
    {"bacterium_id": 5859, "phage_id": 5924},  # this was found in test dataset
    {"bacterium_id": 5859, "phage_id": 5925},
    {"bacterium_id": 5859, "phage_id": 5927},
    {"bacterium_id": 5859, "phage_id": 5928},
    {"bacterium_id": 5859, "phage_id": 5935},
    {"bacterium_id": 5859, "phage_id": 5937},
]

In [6]:
FILTERED_PAIRS = [
    {'bacterium_id': 153, 'phage_id': 2014},
    {'bacterium_id': 153, 'phage_id': 2153},
    {'bacterium_id': 1869, 'phage_id': 6187},
    {'bacterium_id': 5859, 'phage_id': 451},
    {'bacterium_id': 5859, 'phage_id': 5924},
    {'bacterium_id': 5859, 'phage_id': 5928}
]

In [15]:
TEST_PAIR = {"bacterium_id": 153, "phage_id": 2014}

In [36]:
TEST_PAIR = FILTERED_PAIRS[5]

### Data loading section
with 2 possible implementations (  Witihn config / `CUSTOM_RUN`)

In [7]:
if config.input_perphect is not None:
    bacteria_df, phages_df, couples_df = PerphectDataInput(
        input_paths=config.input_perphect  
    ).load()

[INFO] Perphect input files will be read from data/perphect-data/all/bacteria_df.csv, data/perphect-data/all/phages_df.csv and data/perphect-data/all-private-oversampled/couples_df.csv
[INFO] Reading csv files...


### Compiting the embeddings 

compute = config.models, compute CACHED_EMBEDDINGS_OPTION, df, output manager

In [9]:
create_embeddings_bacteria(
    config.bacteria_embedding_models,    # model in config
    config.compute_bacteria_embeddings,  # this is the CACHED_EMBEDDINGS_OPTION
    bacteria_df,                         # from the previous step
    output_manager,                      # ???               
)

create_embeddings_phages(
    config.phages_embedding_models,
    config.compute_phages_embeddings,
    phages_df,
    output_manager,
)

[INFO] Creating embeddings for 3 bacteria models...
[DEBUG] Creating bacteria embeddings for model NT2-TopBottomTruncateStrategy-250M-ov0...


100%|██████████| 231/231 [00:00<00:00, 740.58it/s]

[DEBUG] Saving 231 embeddings for model NT2-TopBottomTruncateStrategy-250M-ov0 to data/embeddings. Shape: torch.Size([1536])


Saving embeddings:   0%|          | 0/231 [00:00<?, ?it/s]

[DEBUG] Creating bacteria embeddings for model MegaDNA-BottomTruncateStrategy-concat-ov0...


...

Saving embeddings:   0%|          | 0/3539 [00:00<?, ?it/s]

[DEBUG] Creating phage embeddings for model DNABERT2-TKPert-concat-J16-g20-ov0-maxlen32768...


100%|██████████| 3539/3539 [00:06<00:00, 582.59it/s]

[DEBUG] Saving 3539 embeddings for model DNABERT2-TKPert-concat-J16-g20-ov0-maxlen32768 to data/embeddings. Shape: torch.Size([12288])


Saving embeddings:   0%|          | 0/3539 [00:00<?, ?it/s]

### Creating or loading the dataset aka meta embeddings

calculating the embeddings

In [8]:
# compute option
bacteria_model_names = [x.name() for x in config.bacteria_embedding_models]
phages_model_names = [x.name() for x in config.phages_embedding_models]

emb_test_data = make_dataset(
    couples_df,                       # got it in pre... previous step ?
    bacteria_model_names,             # here
    phages_model_names,               # here
    output_manager,                   # ???
    DEVICE                            # from initialization phase
    )

[INFO] Creating dataset (loading embeddings)...
[DEBUG] Loading 10300 embeddings for model NT2-TopBottomTruncateStrategy-250M-ov0 from data/embeddings


Loading embeddings:   0%|          | 0/10300 [00:00<?, ?it/s]

[DEBUG] Loading 10300 embeddings for model MegaDNA-BottomTruncateStrategy-concat-ov0 from data/embeddings


Loading embeddings:   0%|          | 0/10300 [00:00<?, ?it/s]

[DEBUG] Loading 10300 embeddings for model DNABERT2-TopBottomTruncateStrategy-ov0-maxlen32768 from data/embeddings


...

[DEBUG] Loading 10300 embeddings for model MegaDNA-MaxStrategy-concat-ov0 from data/embeddings


Loading embeddings:   0%|          | 0/10300 [00:00<?, ?it/s]

[DEBUG] Loading 10300 embeddings for model DNABERT2-TKPert-concat-J16-g20-ov0-maxlen32768 from data/embeddings


Loading embeddings:   0%|          | 0/10300 [00:00<?, ?it/s]

[DEBUG] Final embedding size (bacteria): 4036
[DEBUG] Final embedding size (phages): 14788


alternatively we can load previously computed dataset (embeddings)

In [11]:
# load option
initial_pca_df = torch.load(config.test_path, map_location=DEVICE)

In [10]:
len(initial_pca_df['phage_embedding'])

1426

In [16]:
test_for_saving = emb_test_data[emb_test_data['id'].isin(set(initial_pca_df['id'].values))]
test_unique = emb_test_data[emb_test_data['id'].isin(initial_pca_df['id'].unique())]
#torch.save(test_for_saving, config.test_path)
print(f"test_for_saving: {len(test_for_saving['phage_embedding'])}")
print(f"test_unique: {len(test_unique['phage_embedding'])}")

test_for_saving: 3907
test_unique: 3907


In [ ]:
# Build a set of (bact_id, phage_id) pairs from initial_pca_df
saved_pairs = set(zip(initial_pca_df['bacterium_id'], initial_pca_df['phage_id']))

mask = [
    (b, p) in saved_pairs
    for b, p in zip(emb_test_data['bacterium_id'], emb_test_data['phage_id'])
]
pre_pca_test = emb_test_data[mask]
print(len(pre_pca_test)) 

3907


In [11]:
test_unique2 = emb_test_data.drop_duplicates(subset="id", keep="first").copy()
len(test_unique2['phage_embedding'])

7720

In [17]:
bact_rows = initial_pca_df[initial_pca_df['bacterium_id'] == TEST_PAIR['bacterium_id']]

total     = len(bact_rows)
positive  = (bact_rows['interaction_type'] == 1).sum()
negative  = (bact_rows['interaction_type'] == 0).sum()

print(f"Bacterium {TEST_PAIR['bacterium_id']} in test set:")
print(f"  Total pairs : {total}")
print(f"  Positive    : {positive} ({100 * positive / total:.1f}%)")
print(f"  Negative    : {negative} ({100 * negative / total:.1f}%)")
print()
print(bact_rows[['id', 'phage_id', 'interaction_type']].to_string(index=False))

Bacterium 5859 in test set:
  Total pairs : 583
  Positive    : 252 (43.2%)
  Negative    : 331 (56.8%)

   id  phage_id  interaction_type
 3605      2246                 0
 2494      3557                 1
 3634      4195                 0
 4279      4739                 0
 1491      2660                 1
 3450      4975                 0
 2627      5965                 1
 1317      2496                 1
 3027      3996                 0
 4230      4135                 0
 3595      4754                 0
 3208      2820                 0
 1972      6129                 1
 1389      2561                 1
 2498      3561                 1
 2869      4023                 0
 2788      6073                 0
 3393      4888                 0
 3280      4146                 0
 3482      3998                 0
 2334      5616                 1
 3779      3997                 0
 2103      3203                 1
 2538      3597                 1
 3081      4808                 0
 1647      

In [37]:
print(str(len(initial_pca_df)) + ' lenght of set')
bact_rows = initial_pca_df[initial_pca_df['phage_id'] == TEST_PAIR['phage_id']]

total     = len(bact_rows)
positive  = (bact_rows['interaction_type'] == 1).sum()
negative  = (bact_rows['interaction_type'] == 0).sum()

print(f"Phage {TEST_PAIR['phage_id']} in whole set:")
print(f"  Total pairs : {total}")
print(f"  Positive    : {positive} ({100 * positive / total:.1f}%)")
print(f"  Negative    : {negative} ({100 * negative / total:.1f}%)")
print()
print(bact_rows[['id', 'bacterium_id', 'interaction_type']].to_string(index=False))

1426 lenght of set
Phage 5928 in whole set:
  Total pairs : 1
  Positive    : 1 (100.0%)
  Negative    : 0 (0.0%)

 id  bacterium_id  interaction_type
768          5859                 1


In [30]:
test_for_saving.head()

,id,phage_id,bacterium_id,interaction_type,bacterium_embedding,phage_embedding
0,5365,4968,1804,1,"[tensor(0.3028, device='cuda:0'), tensor(-0.08...","[tensor(0.0849, device='cuda:0'), tensor(-0.10..."
1,5366,4546,1804,1,"[tensor(0.3028, device='cuda:0'), tensor(-0.08...","[tensor(0.0784, device='cuda:0'), tensor(-0.09..."
3,5368,4200,1804,1,"[tensor(0.3028, device='cuda:0'), tensor(-0.08...","[tensor(0.2108, device='cuda:0'), tensor(-0.03..."
4,5369,4942,1804,1,"[tensor(0.3028, device='cuda:0'), tensor(-0.08...","[tensor(0.3448, device='cuda:0'), tensor(-0.10..."
7,5194,4942,5880,1,"[tensor(0.2634, device='cuda:0'), tensor(-0.04...","[tensor(0.3448, device='cuda:0'), tensor(-0.10..."


In [ ]:
print(len(emb_test_data['phage_embedding']))
print(len(initial_pca_df['phage_embedding']))

10300
2060


## New split

Here I wanted to test if I can reproduce the test and check similiarities of my new split with the original one, to dig into the train split as well as test

In [18]:
train, test = train_test_split(emb_test_data, test_size=0.2, random_state=42, shuffle=True)

print(f"train length: {len(train['phage_embedding'])}")
print(f"test lenght: {len(test['phage_embedding'])}")

train length: 8240
test lenght: 2060


Here I assume that probably there is some flaw in the ID structure, so I swith to the key pairs based on bact and phage ID - spoiler alert - I'll get the very same results...

In [19]:
# Build pair sets from both
pairs_test = set(zip(test['bacterium_id'], test['phage_id']))
pairs_saved = set(zip(initial_pca_df['bacterium_id'], initial_pca_df['phage_id']))

print(len(pairs_test))           # how many unique pairs in new test split
print(len(pairs_saved))          # how many in saved
print(len(pairs_test & pairs_saved))  # overlap (intersection)
print(len(pairs_test ^ pairs_saved))  # symmetric diff — pairs in one but not the other

1784
1784
1784
0


In [20]:
pairs_train = set(zip(train['bacterium_id'], train['phage_id']))

print(len(pairs_test))           # how many unique pairs in new test split
print(len(pairs_train))          # how many in saved
print(len(pairs_test & pairs_train))  # overlap (intersection)
print(len(pairs_test ^ pairs_train)) 

1784
6295
359
7361


In [21]:
overlap = (pairs_test & pairs_train)
pairs_test -= overlap 
len(pairs_test)

1425

Quick check for pairs that are present in Federico work (*saved in the binginging*)

In [23]:
filtered_parts = []

for pair in PAIR_LIST:
    key = (pair['bacterium_id'], pair['phage_id'])
    if key in pairs_test:
        filtered_parts.append(pair)

for p in filtered_parts:
    print(p) 

{'bacterium_id': 153, 'phage_id': 2014}
{'bacterium_id': 153, 'phage_id': 2153}
{'bacterium_id': 1869, 'phage_id': 6187}
{'bacterium_id': 5859, 'phage_id': 451}
{'bacterium_id': 5859, 'phage_id': 5924}
{'bacterium_id': 5859, 'phage_id': 5928}


## Creating final df

now time to create proper test dataframe that will be usefull for XAI tests

In [26]:
pair_series = pd.Series(
    list(zip(test['bacterium_id'], test['phage_id'])),
    index=test.index,
)

In [27]:
pair_series

5397    (5180, 5286)
3077    (5244, 5206)
6050    (5178, 5283)
6502    (1869, 2318)
3601    (1804, 4546)
            ...     
221     (5200, 5282)
7383    (5859, 3129)
5991     (527, 5206)
4707    (1758, 4968)
858     (5195, 5315)
Length: 2060, dtype: object

In [29]:
final_test_split = test[pair_series.isin(pairs_test)].copy()
final_test_split.head()

,id,phage_id,bacterium_id,interaction_type,bacterium_embedding,phage_embedding
6502,1127,2318,1869,1,"[tensor(-0.2436, device='cuda:0'), tensor(0.46...","[tensor(-0.4400, device='cuda:0'), tensor(0.56..."
9819,4505,4095,1869,0,"[tensor(-0.2436, device='cuda:0'), tensor(0.46...","[tensor(0.3543, device='cuda:0'), tensor(-0.05..."
932,7056,5303,5196,0,"[tensor(-0.0618, device='cuda:0'), tensor(0.18...","[tensor(-0.0118, device='cuda:0'), tensor(0.18..."
304,6430,5322,5201,0,"[tensor(-0.1654, device='cuda:0'), tensor(0.29...","[tensor(-0.1430, device='cuda:0'), tensor(0.33..."
8927,3605,2246,5859,0,"[tensor(-0.1581, device='cuda:0'), tensor(0.44...","[tensor(-0.1737, device='cuda:0'), tensor(0.38..."


In [30]:
final_test_split.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1426 entries, 6502 to 858
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   1426 non-null   int64 
 1   phage_id             1426 non-null   int64 
 2   bacterium_id         1426 non-null   int64 
 3   interaction_type     1426 non-null   int64 
 4   bacterium_embedding  1426 non-null   object
 5   phage_embedding      1426 non-null   object
dtypes: int64(4), object(2)
memory usage: 78.0+ KB


In [31]:
len(emb_test_data['phage_embedding'].iloc[0])

14788

In [32]:
len(emb_test_data['bacterium_embedding'].iloc[0])

4036

In [33]:
14788 + 4036 == 18824

True

In [34]:
# just in case, this is how df is saved: (may be for custom shapley embeddings)
#
torch.save(final_test_split, config.test_path)

### Loading from the config and appling PCA

In [35]:
emb_test_data = final_test_split.copy()

In [36]:
with open(config.bacteria_pca_path, 'rb') as f:
    pca_bact = pickle.load(f)
emb_test_data["bacterium_embedding"] = list(torch.from_numpy(pca_bact.transform(emb_test_data["bacterium_embedding"].apply(lambda x: x.detach().cpu().numpy()).to_list())).float())  # type: ignore

with open(config.phage_pca_path, 'rb') as f:
    pca_phag = pickle.load(f)
emb_test_data["phage_embedding"] = list(torch.from_numpy(pca_phag.transform(emb_test_data["phage_embedding"].apply(lambda x: x.detach().cpu().numpy()).to_list())).float())  # type: ignore


### Loading model

In [37]:
bacterium_embed_size = len(emb_test_data["bacterium_embedding"].iloc[0]) # always 500 ???
phage_embed_size = len(emb_test_data["phage_embedding"].iloc[0])
model = config.classifier(bacterium_embed_size, phage_embed_size, **config.classifier_params)

model.load_state_dict(torch.load(config.model_path, map_location=DEVICE))

/home/pavel.degterev/miniforge3/envs/pbi/lib/python3.10/site-packages/torch/nn/init.py:412: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


<All keys matched successfully>

### Model prediction

In [39]:
from main import test_model

model.to('cuda') 

test_model(
    emb_test_data, model, batch_size=config.training_config.batch_size, device=DEVICE
)

[INFO] Starting testing...
[INFO] Accuracy (test): 0.9495091438293457
[INFO] Recall (test): 0.9253333210945129
[INFO] F1 score (test): 0.9060052037239075
[INFO] Loss (test): 0.2876856324339983
[INFO] Confusion Matrix (test) (TP, FP, FN, TN): (347, 44, 28, 1007)


(array([[1007,   44],
        [  28,  347]]),
 0.2876856324339983)